In [3]:
# Challenge 1 Group #7: Regression Problem - Aprtment Rental Price Prediction
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)

In [4]:
# loading datasets
train = pd.read_csv("challenge1_train.csv")
test = pd.read_csv("challenge1_test.csv")

train.head()

,ID,price,bathrooms,bedrooms,square_feet,latitude,longitude,category,has_photo,state
0,2,925,1.0,0,116,47.6160,-122.3275,housing/rent/apartment,Thumbnail,WA
1,3,2475,1.0,0,130,40.7629,-73.9885,housing/rent/apartment,Thumbnail,NY
2,5,1695,1.0,0,190,37.7599,-122.4379,housing/rent/apartment,Thumbnail,CA
3,6,1560,1.0,1,200,35.0847,-77.0609,housing/rent/apartment,Thumbnail,NC
4,7,1560,1.0,1,200,35.0960,-77.0272,housing/rent/apartment,Thumbnail,NC


In [5]:
X = train.drop(columns=["price", "ID"]).copy()
y = train["price"].copy()

# seperating training IDs
train_ids = train["ID"].copy()

# preparing test inputs and preserving IDs
X_test = test.drop(columns=["ID"]).copy()
test_ids = test["ID"].copy()

X.head()

,bathrooms,bedrooms,square_feet,latitude,longitude,category,has_photo,state
0,1.0,0,116,47.6160,-122.3275,housing/rent/apartment,Thumbnail,WA
1,1.0,0,130,40.7629,-73.9885,housing/rent/apartment,Thumbnail,NY
2,1.0,0,190,37.7599,-122.4379,housing/rent/apartment,Thumbnail,CA
3,1.0,1,200,35.0847,-77.0609,housing/rent/apartment,Thumbnail,NC
4,1.0,1,200,35.0960,-77.0272,housing/rent/apartment,Thumbnail,NC


In [6]:
# Pipelines for Numerical and Categorical columns
numeric_features = [
    "bathrooms",
    "bedrooms",
    "square_feet",
    "latitude",
    "longitude"
]

categorical_features = [
    "category",
    "has_photo",
    "state"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=False
    ))
])



In [7]:
# Connecting Pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

In [8]:
# checking for every predictor
assert set(numeric_features + categorical_features) == set(X.columns)

assert X.columns.equals(X_test.columns)

print("Preprocessing setup is ready")
print("Training inputs:", X.shape)
print("Training target:", y.shape)
print("Test inputs:", X_test.shape)

Preprocessing setup is ready
Training inputs: (7398, 8)
Training target: (7398,)
Test inputs: (2467, 8)


## Validation Strategy

In [9]:
## continued imports

import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error

In [10]:
## seed & safety check

RANDOM_STATE = 42
assert "ID" not in X.columns and "price" not in X.columns

In [11]:
## hold out split

price_bins = pd.qcut(y, q=10, duplicates="drop")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=price_bins
)

print("Train:", X_train.shape, "| Hold-out:", X_val.shape)

Train: (5918, 8) | Hold-out: (1480, 8)


In [12]:
## checking the split

pd.DataFrame({"train": y_train.describe(), "hold-out": y_val.describe()})

,train,hold-out
count,5918.000000,1480.000000
mean,1460.599020,1471.628378
std,813.573758,853.086628
min,200.000000,288.000000
25%,950.000000,930.000000
50%,1273.500000,1273.500000
75%,1694.750000,1695.000000
max,8800.000000,8500.000000


In [13]:
## folds and scoring

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae":  "neg_mean_absolute_error",
    "r2":   "r2",
}

print([(len(tr), len(va)) for tr, va in cv.split(X_train)])

[(4734, 1184), (4734, 1184), (4734, 1184), (4735, 1183), (4735, 1183)]


In [14]:
## defining functions

def cv_evaluate(name, model):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    res = cross_validate(pipe, X_train, y_train, cv=cv,
                         scoring=scoring, return_train_score=True)
    cv_rmse = -res["test_rmse"].mean()
    train_rmse = -res["train_rmse"].mean()
    return {
        "model": name,
        "CV RMSE (mean)": cv_rmse,
        "CV RMSE (std)": res["test_rmse"].std(),
        "Train RMSE": train_rmse,
        "Gap (CV - Train)": cv_rmse - train_rmse,
        "CV MAE": -res["test_mae"].mean(),
        "CV R2": res["test_r2"].mean(),
    }

def compare(models):
    rows = [cv_evaluate(n, m) for n, m in models.items()]
    return pd.DataFrame(rows).sort_values("CV RMSE (mean)").reset_index(drop=True)

def holdout_rmse(model):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    return np.sqrt(mean_squared_error(y_val, pipe.predict(X_val)))

In [15]:
## final baseline checks

baseline = compare({"Baseline (mean price)": DummyRegressor(strategy="mean")})
print("Std of price in X_train:", y_train.std())
baseline

Std of price in X_train: 813.5737582900422


/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,model,CV RMSE (mean),CV RMSE (std),Train RMSE,Gap (CV - Train),CV MAE,CV R2
0,Baseline (mean price),812.320176,45.528834,813.408304,-1.088128,546.526106,-0.000683


### How to use this validation setup
- Evaluate models with `compare({"Name": model, ...})`; give every model `random_state=RANDOM_STATE`.

- Grid search: `GridSearchCV(Pipeline([("prep", preprocessor), ("model", ...)]), param_grid={"model__<param>": [...]}, cv=cv, scoring="neg_root_mean_squared_error")`, fit on `X_train, y_train` only.
  
- Use `X_val` / `y_val` only once, via `holdout_rmse(final_model)`, after the final model is chosen.
  
- Before predicting `X_test`, refit the final model on all of `X, y`.

## Model Imports

In [16]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}
 
results = compare(models)
results

/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zero

,model,CV RMSE (mean),CV RMSE (std),Train RMSE,Gap (CV - Train),CV MAE,CV R2
0,Hist Gradient Boosting,410.618559,19.437216,310.130645,100.487915,255.957076,0.743313
1,Random Forest,426.912457,17.711397,160.924261,265.988196,261.506247,0.722259
2,Gradient Boosting,454.117941,23.243680,391.865331,62.252610,292.142887,0.687048
3,Linear Regression,518.504486,32.725760,510.861495,7.642991,336.079274,0.592315
4,Decision Tree,563.900533,14.378524,36.987003,526.913530,338.182132,0.513478


## Interpretation

- Hist Gradien Boosting was the best model with the lowest CV RMSE at about (412)

- As tree-based models outperformed linar and single tree models, it's safe to say that the data relationship between prices, features and locations are non-linar.

In [17]:
# Improving the models
from sklearn.model_selection import GridSearchCV
from sklearn.base import clone

hgb_pipeline = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", HistGradientBoostingRegressor(random_state=RANDOM_STATE))
])

In [18]:
hgb_param_grid = {
    "model__learning_rate": [0.05, 0.1],
    "model__max_iter": [100, 300],
    "model__max_leaf_nodes": [15, 31],
    "model__l2_regularization": [0.0, 1.0]
}

In [19]:
hgb_search = GridSearchCV(
    estimator=hgb_pipeline,
    param_grid=hgb_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=1
)

hgb_search.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zero

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__l2_regularization': [0.0, 1.0], 'model__learning_rate': [0.05, 0.1], 'model__max_iter': [100, 300], 'model__max_leaf_nodes': [15, 31]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``

In [20]:
hgb_best_rmse = -hgb_search.best_score_

print("Best settings:", hgb_search.best_params_)
print(f"Best CV RMSE: ${hgb_best_rmse:,.2f}")

Best settings: {'model__l2_regularization': 0.0, 'model__learning_rate': 0.1, 'model__max_iter': 300, 'model__max_leaf_nodes': 15}
Best CV RMSE: $403.09


In [22]:
rf_pipeline = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=1
    ))
])

In [23]:
rf_param_grid = {
    "model__max_depth": [None, 15],
    "model__min_samples_leaf": [1, 3, 5],
    "model__max_features": [0.8, 1.0]
}


In [24]:
rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=1
)

rf_search.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zero

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [None, 15], 'model__max_features': [0.8, 1.0], 'model__min_samples_leaf': [1, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_

In [25]:
rf_best_rmse = -rf_search.best_score_

print("Best Random Forest settings:", rf_search.best_params_)
print(f"Random Forest CV RMSE: ${rf_best_rmse:,.2f}")
print(f"Hist Gradient Boosting CV RMSE: ${hgb_best_rmse:,.2f}")

Best Random Forest settings: {'model__max_depth': None, 'model__max_features': 0.8, 'model__min_samples_leaf': 1}
Random Forest CV RMSE: $423.21
Hist Gradient Boosting CV RMSE: $403.09


In [27]:
hgb_index = hgb_search.best_index_
rf_index = rf_search.best_index_

tuned_results = pd.DataFrame({
    "model": ["Tuned Hist Gradient Boosting", "Tuned Random Forest"],
    "CV RMSE (mean)": [hgb_best_rmse, rf_best_rmse],
    "Train RMSE": [
        -hgb_search.cv_results_["mean_train_score"][hgb_index],
        -rf_search.cv_results_["mean_train_score"][rf_index]
    ]
})

tuned_results["Gap (CV - Train)"] = (
    tuned_results["CV RMSE (mean)"] - tuned_results["Train RMSE"]
)

tuned_results.round(2)

,model,CV RMSE (mean),Train RMSE,Gap (CV - Train)
0,Tuned Hist Gradient Boosting,403.09,291.30,111.79
1,Tuned Random Forest,423.21,159.44,263.77


In [33]:
# selecting model
comparison_columns = [
    "model", "CV RMSE (mean)", "Train RMSE", "Gap (CV - Train)"
]

comparison = pd.concat(
    [results[comparison_columns], tuned_results[comparison_columns]],
    ignore_index=True
)

comparison = comparison.sort_values("CV RMSE (mean)").reset_index(drop=True)

comparison.round(2)

selected_model = hgb_search.best_estimator_

holdout_predictions = selected_model.predict(X_val)

final_holdout_rmse = np.sqrt(
    mean_squared_error(y_val, holdout_predictions)
)

print(f"Selected model CV RMSE: ${hgb_best_rmse:,.2f}")
print(f"Final holdout RMSE: ${final_holdout_rmse:,.2f}")

Selected model CV RMSE: $403.09
Final holdout RMSE: $427.01


# Selected Model: Tuned Hist Gradient Boosting
- Reason: Lowest Mean CV RSME at 403.09
- Settings: Learning rate at 0.1 with 300 boosting iterations and max 15 leaves per tree. L2 regularization at 0.0.
- Overfitting: The Training RMSE at 291.20 and CV RMSE at 403.09 show a gap of 111.79, much smaller than the 263.77 gap from RF
- Tradeoff: Harder to interpret than a Linear Regression, but achieved very good validation performance.
- Final evaluation: Holdout RMSE 427.01 on 1,480 rows excluded from training and tuning.

In [36]:
# Generating predictions
final_model = clone(selected_model)

final_model.fit(X, y)

print(f"Final model trained on {len(X):,} labeled rows.")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Final model trained on 7,398 labeled rows.


In [37]:
test_predictions = final_model.predict(X_test)

submission = pd.DataFrame({
    "ID": test_ids,
    "price": test_predictions
})

submission.head()

/opt/anaconda3/envs/stock-risk/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,ID,price
0,1,1146.866632
1,4,2154.395539
2,9,972.755858
3,11,618.193556
4,13,644.750561


In [38]:
assert submission.shape == (2467, 2)
assert submission.columns.tolist() == ["ID", "price"]
assert submission["ID"].equals(test["ID"])
assert submission["ID"].is_unique
assert np.isfinite(submission["price"]).all()

print("All prediction checks passed.")

submission["price"].describe()


All prediction checks passed.


count    2467.000000
mean     1469.757852
std       766.521314
min       481.442997
25%      1006.049834
50%      1256.682422
75%      1639.326571
max      7241.962183
Name: price, dtype: float64

In [39]:
submission.to_csv("David_predictions.csv", index=False)

print("Saved to:", Path("David_predictions.csv").resolve())

Saved to: /Users/tef/Documents/School/Fall 2026/Intro Machine Learning/Challenge 1/CAI_4105_2026_Fall_Team_Seven/David_predictions.csv
